In [ ]:
import warnings
warnings.filterwarnings('ignore')
import json
import pandas as pd
from numpy import round, isnan, NaN
from tqdm import tqdm
from utils.html2text import html2text
from utils.moodle_connection import moodle_connection, retrieve_data_from_MOODLE
from utils.neo4j_connection import neo4j_connection
from utils.utils import assignment_4C, available_metrics, text_preprocess
from utils.translations import d_eval_questionnaire

# Neo4j credentials
with open("Resources/neo4j_settings.json", "r") as file:
    neo4j_settings = json.load(file)
# MOODLE credentials
moodle_settings = {
    "host": "...",
    "user": "...",
    "password": "...",
    "port": 3306,
    "database": "... ",
}

# Course Ids
course_ids = (...)

def get_organization(x):
    return "Demo"

def get_language(x):
    return "english"


moodle_url = "..."
# Connection with Neo4j and SQL server
graph = neo4j_connection(neo4j_settings=neo4j_settings, clean_graph=False)
connection = moodle_connection(moodle_settings)
cursor = connection.cursor()


### Courses

In [ ]:
query = """SELECT 
    c.id AS course_id,
    c.shortname AS course_shortname,
    c.fullname AS course_fullname,
    from_unixtime(c.timecreated) AS timecreated,
    c.summary AS summary
FROM 
    mdl_course c
LEFT JOIN 
    mdl_logstore_standard_log l ON l.courseid = c.id AND l.action = 'created' AND l.target = 'course'
GROUP BY 
    c.id;"""
        
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
print("[INFO] Number of records: ", df.shape[0])

# Get course Ids
if course_ids is not None:
    df = df[df['course_id'].isin(course_ids)]
else:
    course_ids = tuple(df['course_id'].unique())

for idx in tqdm(df.index):
    course_id = df.loc[idx]["course_id"]
    course_name = df.loc[idx]["course_fullname"].strip()
    course_description = html2text(df.loc[idx]["summary"])
    course_timecreated = df.loc[idx]["timecreated"]
    
    graph.query(f"""MERGE (c:COURSE {{id:{course_id}, title:"{course_name}", description:"{course_description}", timecreated:"{course_timecreated}", organization:"{get_organization(course_id)}"}})""")
print("[INFO] Courses were imported")

### Users (Learners & Teachers)

In [ ]:
query = f"""SELECT 
    u.id AS id,
    u.username AS username,
    u.email AS email,
    u.institution AS institution,
    u.country AS country,
    u.confirmed AS confirmed,
    r.shortname AS role,
    cse.id AS course_id
FROM 
    mdl_user u
JOIN 
    mdl_role_assignments ra ON u.id = ra.userid
JOIN 
    mdl_context c ON ra.contextid = c.id
JOIN 
    mdl_role r ON ra.roleid = r.id
JOIN 
    mdl_course cse ON c.instanceid = cse.id AND c.contextlevel = 50
WHERE 
    cse.id IN {course_ids}
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).drop_duplicates()
print("[INFO] Number of records: ", df.shape[0])


# Sanity check
df = df[df["confirmed"] == 1]
# Processing
df_roles = (
    df.groupby(["id", "username", "email", "course_id"]).agg(lambda x: list(set(x))).reset_index()[["id", "username", "course_id", "email", "role"]]
)
df_roles["role"] = df_roles["role"].apply(
    lambda x: "learner" if x == ["student"] else "viewer" if "augmentor_partner" in x else "teacher"
)

# Include data to Neo4j
for idx in tqdm(df_roles.index):
    user_id = df_roles.iloc[idx]["id"]
    username = df_roles.iloc[idx]["username"]
    email = df_roles.iloc[idx]["email"]
    course_id = df_roles.iloc[idx]["course_id"]
    role = df_roles.iloc[idx]["role"].upper()
    
    query = f"""MATCH (c:COURSE)
    WHERE c.id = {course_id}
    MERGE (l:{role} {{user_id:{user_id}, username:"{username}", email:"{email}", organization:"{get_organization(course_id)}"}})
    MERGE (l)-[:REGISTERED]->(c)"""
    graph.query(query)
        
print("[INFO] Learners, Viewers and Teachers imported and registed to the corresponding courses")



# Get Learners' IDs
learner_ids = graph.query(f"match (l:LEARNER)-[]->(c:COURSE) where c.id in {list(course_ids)} return collect(distinct l.user_id) as usernames")[0]['usernames']
print("[INFO] Number of learner IDs: ", len(learner_ids))

### Courses and Modules

In [ ]:
for course_id in course_ids:
    query = f"""SELECT 
        id AS section_id,
        course AS course_id,
        name AS title,
        section AS section_name,
        sequence
    FROM 
        mdl_course_sections
    WHERE 
        course = {course_id}
    ORDER BY 
        sequence;
    """
    # Fetch records
    rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
    # Convert records to DataFrame    
    df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
    # Processing
    df = df.drop('sequence', axis=1).drop_duplicates()
    df['section_name'] += 1
    df['title'] = df['title'].apply(lambda x: x.strip() if x is not None else x)
    print(f"[INFO] Course Id: {course_id:.0f} | Number of records: ", df.shape[0])

    for idx in tqdm(df.index):
        section_id = df.loc[idx]["section_id"]
        title = df.loc[idx]["title"]
        section_name = f"MODULE:{df.loc[idx]['section_name']}"
        course_id = df.loc[idx]["course_id"]
        URL = f"{moodle_url}/course/section.php?id={section_id}"

        query = f"""MATCH (c:COURSE)
                    WHERE c.id = {course_id} AND c.organization = "{get_organization(course_id)}"
                    MERGE (m:MODULE {{id:{section_id}, title:"{title}", code:"{section_name}", organization:"{get_organization(course_id)}", url:"{URL}"}})
                    MERGE (c)-[:HAS_MODULE]->(m)"""

        graph.query(query)

print("[INFO] Modules of each course were created")  

### Activities

For each module include its activities

In [ ]:
for course_id in course_ids:
    query = f"""SELECT 
        cs.id AS section_id,
        cs.course AS course_id,
        f.name as title,
        'Forum' AS activity_type,
        f.id AS activity_id,
        f.intro AS description,
        CONCAT('{moodle_url}/mod/forum/view.php?id=', cm.id) AS URL
    FROM 
        mdl_course_sections cs
    LEFT JOIN 
        mdl_course_modules cm ON cm.section = cs.id
    LEFT JOIN 
        mdl_forum f ON f.course = cs.course AND cm.instance = f.id AND cm.module = (SELECT id FROM mdl_modules WHERE name = 'forum')
    WHERE 
        cs.course = {course_id}
           
    UNION ALL

    SELECT 
        cs.id AS section_id,
        cs.course AS course_id,
        q.name AS title,
        'Quiz' AS activity_type,
        q.id AS activity_id,
        q.intro AS description,
        CONCAT('{moodle_url}/mod/quiz/view.php?id=', cm.id) AS URL
    FROM 
        mdl_course_sections cs
    LEFT JOIN 
        mdl_course_modules cm ON cm.section = cs.id
    LEFT JOIN 
        mdl_quiz q ON q.course = cs.course AND cm.instance = q.id AND cm.module = (SELECT id FROM mdl_modules WHERE name = 'quiz')
    WHERE 
        cs.course = {course_id}

    UNION ALL

    SELECT 
        cs.id AS section_id,
        cs.course AS course_id,
        a.name AS title,
        'Assign' AS activity_type,
        a.id AS activity_id,
        a.intro AS description,
        CONCAT('{moodle_url}/mod/assign/view.php?id=', cm.id) AS URL
    FROM 
        mdl_course_sections cs
    LEFT JOIN 
        mdl_course_modules cm ON cm.section = cs.id
    LEFT JOIN 
        mdl_assign a ON a.course = cs.course AND cm.instance = a.id AND cm.module = (SELECT id FROM mdl_modules WHERE name = 'assign')
    WHERE 
        cs.course = {course_id}
        
    UNION ALL

    SELECT 
        cs.id AS section_id,
        cs.course AS course_id,
        s.name AS title,
        'Scorm' AS activity_type,
        s.id AS activity_id,
        s.intro AS description,
        CONCAT('{moodle_url}/mod/scorm/view.php?id=', cm.id) AS URL
    FROM 
        mdl_course_sections cs
    LEFT JOIN 
        mdl_course_modules cm ON cm.section = cs.id
    LEFT JOIN 
        mdl_scorm s ON s.course = cs.course AND cm.instance = s.id AND cm.module = (SELECT id FROM mdl_modules WHERE name = 'scorm')
    WHERE 
        cs.course = {course_id}

    ORDER BY 
        section_id, activity_type, activity_id;
    """

    # Fetch records
    rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
    # Convert records to DataFrame
    df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])

    print(f"[INFO] Course Id: {course_id:.0f} | Number of records: ", df.shape[0])
    # Processing
    df.dropna(inplace=True, ignore_index=True)
    df['description'] = df['description'].apply(html2text)
    for idx in tqdm(df.index):
        section_id = df.loc[idx]["section_id"]
        activity_type = df.loc[idx]["activity_type"]
        activity_id = int(df.loc[idx]["activity_id"])
        activity_title = text_preprocess(df.loc[idx]["title"])
        description = text_preprocess(df.loc[idx]["description"])
        id = f"{activity_type.upper()}:{activity_id}"
        url = df.loc[idx]["URL"]
        
        
        query = f"""MATCH (c:COURSE)-[]->(m:MODULE)
                    WHERE c.id = {course_id} AND m.id = {section_id} AND m.organization = "{get_organization(course_id)}"
                    MERGE (a:ACTIVITY {{id:"{id}", type:"{activity_type}", title:"{activity_title}", organization:"{get_organization(course_id)}", description:"{description}", url:"{url}"}})
                    MERGE (m)-[:HAS_ACTIVITY]->(a)"""

        graph.query(query)

print("[INFO] Activities of each Module were created")



# Get course information (Module and Activities)
query = f"""
MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY)
WHERE c.id IN {list(course_ids)}
WITH c.id AS course_id, m.code AS module_code, collect(a.id) AS activity_ids
WITH course_id, collect({{code: module_code, activity_ids: activity_ids}}) AS modules
RETURN collect({{course_id: course_id, modules: modules}}) AS result
"""
courses_info = graph.query(query)[0][0]

In [ ]:
# Get rubrics max scores
query = """SELECT 
    rls.criterionid AS rubric_criterion_id,
    max(rls.score) AS max_level_score
FROM
    mdl_gradingform_rubric_levels AS rls
group by 
    rls.criterionid
"""
# Retrieve user responses from the Moodle database
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
df_rubric_scores = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).drop_duplicates()

#### Activity: FORUM

##### Include FORUM's description

In [ ]:
query = f"""SELECT 
    f.id AS forum_id,
    f.course AS course_id,
    f.type AS forum_type,
    f.name AS forum_name,
    f.intro AS forum_intro
FROM 
    mdl_forum f
WHERE
    f.course in {course_ids};"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
print("[INFO] Number of records: ", df.shape[0])
# Processing
df["forum_intro"] = df["forum_intro"].apply(html2text)

for idx in tqdm(df.index):
    forum_id = df.loc[idx]["forum_id"]
    forum_type = text_preprocess(df.loc[idx]["forum_type"])
    forum_name = text_preprocess(df.loc[idx]["forum_name"])
    forum_intro = text_preprocess(df.loc[idx]["forum_intro"])
    course_id = df.loc[idx]["course_id"]
    
    query = f"""MATCH (a:ACTIVITY)
                WHERE a.id = "FORUM:{forum_id}" AND a.organization = "{get_organization(course_id)}"
                SET a.forum_type = "{forum_type}", a.title = "{forum_name}", a.description = "{forum_intro}"
             """
    graph.query(query)
print("[INFO] FORUM properties were updated")

##### Retrieve Rubrics

In [ ]:
# Step 1. Get learner's performance based on Rubric
query = f"""SELECT
    f.id AS forum_id,
    gi.itemid as itemid,
    f.name AS forum_name,
    f.course as course_id,
    rc.id AS rubric_criterion_id,
    rls.definition AS level_definition,
    grf.levelid AS levelid,
    rls.score AS level_score,
    rc.description AS description
FROM
    mdl_forum AS f
JOIN
    mdl_course_modules AS cm ON cm.instance = f.id AND cm.module = (
        SELECT id FROM mdl_modules WHERE name = 'forum'
    )
JOIN
    mdl_context AS ctx ON ctx.instanceid = cm.id AND ctx.contextlevel = 70
JOIN
    mdl_grading_areas AS ga ON ga.contextid = ctx.id
JOIN
    mdl_grading_definitions AS gd ON gd.areaid = ga.id AND gd.method = 'rubric'
JOIN
    mdl_gradingform_rubric_criteria AS rc ON rc.definitionid = gd.id
JOIN
    mdl_gradingform_rubric_levels AS rls ON rls.criterionid = rc.id
JOIN
    mdl_grading_instances AS gi ON gi.definitionid = gd.id
JOIN
    mdl_gradingform_rubric_fillings AS grf ON grf.instanceid = gi.id AND grf.criterionid = rc.id
WHERE
    grf.levelid = rls.id and f.course in {course_ids}
ORDER BY 
     f.id;"""
# Retrieve user responses from the Moodle database
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
df_rubrics = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).drop_duplicates()
df_rubrics["level_definition"] = df_rubrics["level_definition"].apply(lambda x: x.replace('\r', '').replace('\n', ''))
df_rubrics['level_definition'] = df_rubrics['level_definition'].apply(
    lambda x: (x.strip() + ".") if x and not x.strip().endswith(".") else x.strip()
)
print("[INFO] (Rubrics) Number of records: ", df_rubrics.shape[0])
# Normalization
df_rubrics = pd.merge(df_rubrics, df_rubric_scores,on=["rubric_criterion_id"], how="left")
df_rubrics['level_score'] /= df_rubrics['max_level_score']
# Include 4Cs
df_rubrics['metric'] = df_rubrics['description'].apply(assignment_4C)

# Step 2. Retrieve grades
query = "SELECT forum AS forum_id,userid AS user_id,id AS itemid,grade AS grade from mdl_forum_grades"
# Retrieve user responses from the Moodle database
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
df_grades = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).drop_duplicates().dropna(ignore_index=True)
print("[INFO] (Learner grades) Number of records: ", df_grades.shape[0])

# Step 3. Include Rubrics along with the corresponding grade & Include 4C grades
for metric in available_metrics: 
    if metric == "Cognitive": continue
    df_grades[metric] = None
df_grades['Creativity_reasoning'] = None
df_grades['Collaboration_reasoning'] = None
df_grades['Critical thinking_reasoning'] = None
df_grades['Communication_reasoning'] = None
df_grades['Cognitive_reasoning'] = None
for idx in df_grades.index:
    # Get rubrics per metric
    temp_df = df_rubrics[df_rubrics['itemid'] == df_grades['itemid'][idx]][['level_definition', 'metric']].groupby('metric').agg(list).reset_index()
    temp_df['level_definition'] = temp_df['level_definition'].apply(lambda L: text_preprocess(" ".join([f"{x}" for x in L if len(x) > 10])))
    temp_df['level_definition'] = temp_df['level_definition'].apply(lambda x: x.replace("'",""))
    rubrics = dict(zip(temp_df['metric'].str.strip(), temp_df['level_definition'].str.strip()))
    
    # Get 4Cs grades
    d = df_rubrics[(df_rubrics['itemid'] == df_grades['itemid'][idx])][['metric', 'level_score']].groupby('metric').mean().to_dict()['level_score']
    
    for metric, grade in d.items():
        df_grades.loc[idx, f"{metric}_reasoning"] = rubrics[metric]
        if metric == "Cognitive": continue
        df_grades.loc[idx, metric] = 100.0 * grade

##### Assign Learners to FORUMs

In [ ]:
query = f"""SELECT 
    p.userid AS user_id,
    f.id AS forum_id,
    f.course AS course_id,
    MAX(fg.grade) AS Cognitive
FROM 
    mdl_forum_posts p
JOIN 
    mdl_forum_discussions d ON p.discussion = d.id
JOIN 
    mdl_forum f ON d.forum = f.id
LEFT JOIN 
    mdl_forum_grades fg ON f.id = fg.forum AND p.userid = fg.userid
WHERE
    f.course IN {course_ids}
GROUP BY 
    p.userid, f.id;"""
# Note: Actions is not available; thus, number_of_submissions = NaN
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df = df[df["user_id"].isin(learner_ids)]
print("[INFO] Number of records: ", df.shape[0])
# Include rubrics
df = pd.merge(df, df_grades[['user_id', 'forum_id', 'Creativity', 'Critical thinking', 'Collaboration', 'Communication', 'Creativity_reasoning', 'Collaboration_reasoning', 'Critical thinking_reasoning', 'Communication_reasoning', 'Cognitive_reasoning']], on=['user_id', 'forum_id'], how='left')
# Preprocess
for metric in available_metrics:
    df[metric] = df[metric].astype('float')
    df[metric] = df[metric].apply(lambda x: x if isnan(x) else round(x,1))
df["Cognitive"] = df["Cognitive"].apply(lambda x: NaN if x < 0 else x)

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    forum_id = df.loc[idx]["forum_id"]
    course_id = df.loc[idx]['course_id']
    # Include metrics & their corresponding rubrics
    metrics_input, rubrics = [], []
    for metric in available_metrics:
        if df.loc[idx][metric] and not isnan(df.loc[idx][metric]):
            metrics_input += [f"{metric.replace(' ','_')}_grade:{df.loc[idx][metric]}"]
            if df.loc[idx][f"{metric}_reasoning"]:
                rubrics += [f"{metric.replace(' ','_')}_reasoning:'{df.loc[idx][f'{metric}_reasoning']}'"]
    features = ", ".join(metrics_input + rubrics)
        
    query = f"""MATCH (l:LEARNER) 
            WHERE l.user_id={user_id} AND l.organization = "{get_organization(course_id)}"
            MATCH (a:ACTIVITY) 
            WHERE a.id="FORUM:{forum_id}" AND a.organization = "{get_organization(course_id)}"
            MERGE (l)-[:PARTICIPATE {{ {features} }}]->(a)"""
    graph.query(query)

print("[INFO] Learners were assigned to FORUMs")

##### Engagement information

In [ ]:
query = f"""SELECT
    fp.userid AS user_id,
    u.username AS username,
    fd.forum AS forum_id,
    fd.course AS course_id,
    fp.discussion AS discussion_id,
    l.action AS action,
    l.id,
    l.timecreated AS action_time_unix,
    FROM_UNIXTIME(l.timecreated) AS action_time_readable
FROM
    mdl_forum_posts fp
JOIN
    mdl_forum_discussions fd ON fp.discussion = fd.id
JOIN
    mdl_forum f ON fd.forum = f.id
LEFT JOIN
    mdl_logstore_standard_log l
        ON l.objectid = fp.id
        AND l.userid = fp.userid
        AND l.component = 'mod_forum'
LEFT JOIN
    mdl_user u ON u.id = fp.userid
WHERE
    fd.course IN {course_ids}
ORDER BY l.timecreated"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
# Convert records to DataFrame
df_engagement = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).dropna()
# Pre-processing
df_engagement = df_engagement[df_engagement["user_id"].isin(learner_ids)]
print("[INFO] Number of records: ", df_engagement.shape[0])

In [ ]:
from utils.engagement_forum import generate_forum_statistics
import warnings
warnings.filterwarnings('ignore')

# Iterate through each course in the courses_info data structure
for course_item in courses_info:
    # Extract course ID from current course item
    course_id = course_item["course_id"]
    print(f"[INFO] Course ID: {course_id}")
    if course_id in (2, 4, 12):
        pilot = "IASIS"
    elif course_id in (10, 11):
        pilot = "EASD"
    elif course_id in (16, 33):
        pilot = "UPAT"
    elif course_id == 24:
        pilot = "KTU"
    elif course_id == 34:
        pilot = "Demo"
    else:
        raise ValueError(f"Course with ID: {course_id} does not belong to any pilot")

    # Process each module within the current course
    for module_item in tqdm(course_item["modules"]):
        # Extract module ID and initialize empty list for FORUM IDs
        module_code, forum_ids = module_item["code"], []

        # Filter activity IDs to find only forum activities
        for id in module_item["activity_ids"]:
            if "FORUM" in id:
                # Extract numeric forum ID from the string format (e.g., "FORUM:123" -> 123)
                forum_ids.append(int(id.split(":")[-1]))

        # Process module-level forum statistics if forums exist
        if not forum_ids: continue        
        try:
            # Generate statistics for all forums in this module
            learner_stats, learner_data, module_stats, module_data = generate_forum_statistics(df=df_engagement, pilot=pilot, module_id=module_code, forum_ids=forum_ids, course_id=course_id, graph=graph, moodle_settings=moodle_settings, cursor=cursor)
        except Exception as e:
            print(f"[WARNING] Error occured for '{module_code}' with Forum IDs: {forum_ids}")
            print(f"Reason: {e}")
            continue

        # Update module node in graph database with engagement analysis
        graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) 
                    where m.code = '{module_code}' and c.id = {course_id}
                    set m.statistics = coalesce(m.statistics,'') + '{module_stats}\n\n'""")

        # Process individual learner statistics for module participation
        for username in learner_stats:
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) 
                    MATCH (l:LEARNER)-[]->(c) 
                    where l.username = '{username}' and m.code = '{module_code}' and c.id = {course_id}
                    MERGE (l)-[r:PARTICIPATE]->(m)
                    set r.engagement = coalesce(r.engagement,'') + '{learner_stats[username]}\n\n'""")

        # Process individual forum statistics (Learner -> Forum relationships)
        for forum_id in forum_ids:

            try:
                # Generate statistics for individual forum
                learner_stats, learner_data, activity_stats, activity_data = generate_forum_statistics(
                    df=df_engagement, pilot=pilot, module_id=None, forum_ids=[forum_id], course_id=course_id, graph=graph, moodle_settings=moodle_settings, cursor=cursor
                )
            except Exception as e:
                print(f"[WARNING] Error occured for Forum ID: {forum_id}")
                print(f"Reason: {e}")
                continue

            # Update activity node with engagement analysis
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY) 
                        where m.code = '{module_code}' and a.id = 'FORUM:{forum_id}' and c.id = {course_id}
                        set a.statistics = '{activity_stats}'""")
            
            # Update learner-activity relationships with individual engagement data
            for username in learner_stats:
                graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY)
                            where c.id = {course_id} and m.code = '{module_code}' and a.id = 'FORUM:{forum_id}'
                            MATCH (l:LEARNER)-[r]->(a)
                            where l.username = '{username}'
                            set r.engagement = '{learner_stats[username]}',
                                r.number_of_posts = {learner_data[username]['total_posts']},
                                r.number_of_clicks = {learner_data[username]['total_clicks']},
                                r.number_of_actions = {learner_data[username]['total_actions']},
                                r.number_of_discussions = {learner_data[username]['posting_sessions']},
                                r.time = {learner_data[username]['total_time_spent']},
                                r.avg_time_per_attempt = {learner_data[username]['avg_time_per_session']}""")

#### Activity: QUIZ

##### Include QUIZ's description

In [ ]:
query = f"""SELECT 
    q.id AS quiz_id,
    q.course AS course_id,
    q.name AS quiz_name,
    q.intro AS quiz_intro
FROM 
    mdl_quiz q
WHERE
    q.course in {course_ids};"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
print("[INFO] Number of records: ", df.shape[0])
# Processing
df["quiz_intro"] = df["quiz_intro"].apply(html2text)

for idx in tqdm(df.index):
    quiz_id = df.loc[idx]["quiz_id"]
    quiz_name = text_preprocess(df.loc[idx]["quiz_name"])
    quiz_intro = text_preprocess(df.loc[idx]["quiz_intro"])
    course_id = df.loc[idx]['course_id']
    query = f"""MATCH (a:ACTIVITY)
                WHERE a.id = "QUIZ:{quiz_id}" AND a.organization = "{get_organization(course_id)}"
                SET a.title = "{quiz_name}", a.description = "{quiz_intro}"
             """

    graph.query(query)

print("[INFO] QUIZ properties were updated")

##### Assign Learners to Quizzes

In [ ]:
query = f"""SELECT 
    qa.userid AS user_id,
    q.id AS quiz_id,
    q.course AS course_id,
    MAX(qg.grade) AS user_grade,
    q.grade AS quiz_max_grade
FROM 
    mdl_quiz q
JOIN 
    mdl_quiz_attempts qa ON q.id = qa.quiz
JOIN 
    mdl_quiz_grades qg ON q.id = qg.quiz AND qa.userid = qg.userid
WHERE
    q.course IN {course_ids}
GROUP BY 
    qa.userid, q.id;
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
# Preprocess
df["quiz_max_grade"] = df["quiz_max_grade"].apply(lambda x: x if x > 0 else None).astype('float')
df["user_grade"] = df["user_grade"].astype('float')
df["user_grade"] = df.apply(lambda row: row['user_grade'] if isnan(row['quiz_max_grade']) else round(100.0*row['user_grade']/row['quiz_max_grade'], 1), axis=1)
df = df[df["user_id"].isin(learner_ids)]
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    quiz_id = int(df.loc[idx]["quiz_id"])
    course_id = df.loc[idx]['course_id']
    # Get time and grade
    user_grade = f"Cognitive_grade:{df.loc[idx]['user_grade']}" if df.loc[idx]['user_grade'] >= 0 else ""

    query = f"""MATCH (l:LEARNER) 
                WHERE l.user_id={user_id} AND l.organization = "{get_organization(course_id)}"
                MATCH (a:ACTIVITY) 
                WHERE a.id="QUIZ:{quiz_id}" AND a.organization = "{get_organization(course_id)}"
                MERGE (l)-[:PARTICIPATE {{ {user_grade} }}]->(a)"""
    graph.query(query)
    
print("[INFO] Learners were assigned to QUIZs")

##### Engagement information

In [ ]:
query = f"""SELECT 
    qa.userid AS user_id,
    u.username AS username,
    q.id AS quiz_id,
    q.course AS course_id,
    l.action as action,
    l.id,
    l.timecreated AS action_time_unix,
    FROM_UNIXTIME(l.timecreated) AS action_time_readable
FROM 
    mdl_quiz q
JOIN 
    mdl_quiz_attempts qa ON q.id = qa.quiz
LEFT JOIN 
    mdl_logstore_standard_log l ON l.objectid = qa.id AND l.userid = qa.userid AND l.component = 'mod_quiz'
LEFT JOIN
    mdl_user u ON u.id = qa.userid
WHERE
    q.course IN {course_ids}
ORDER BY l.timecreated
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
# Convert records to DataFrame
df_engagement = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).dropna()
# Pre-processing
df_engagement = df_engagement[df_engagement["user_id"].isin(learner_ids)]
print("[INFO] Number of records: ", df_engagement.shape[0])

In [ ]:
from utils.engagement_quiz import generate_quiz_statistics
import warnings
warnings.filterwarnings('ignore')

# Iterate through each course in the courses_info data structure
for course_item in courses_info:
    # Extract course ID from current course item
    course_id = course_item["course_id"]
    print(f"[INFO] Course ID: {course_id}")
    if course_id in (2, 4, 12):
        pilot = "IASIS"
    elif course_id in (10, 11):
        pilot = "EASD"
    elif course_id in (16, 33):
        pilot = "UPAT"
    elif course_id == 24:
        pilot = "KTU"
    elif course_id == 34:
        pilot = "Demo"        
    else:
        raise ValueError(f"Course with ID: {course_id} does not belong to any pilot")

    # Process each module within the current course
    for module_item in tqdm(course_item["modules"]):
        # Extract module ID and initialize empty list for quiz IDs
        module_code, quiz_ids = module_item["code"], []

        # Filter activity IDs to find only quiz activities
        for id in module_item["activity_ids"]:
            if "QUIZ" in id:
                # Extract numeric quiz ID from the string format (e.g., "QUIZ:123" -> 123)
                quiz_ids.append(int(id.split(":")[-1]))

        # Process module-level quiz statistics if quizzes exist
        if not quiz_ids: continue        
        try:
            # Generate statistics for all quizzes in this module
            learner_stats, learner_data, module_stats, module_data = generate_quiz_statistics(df=df_engagement, pilot=pilot, module_id=module_code, quiz_ids=quiz_ids, course_id=course_id, graph=graph, moodle_settings=moodle_settings, cursor=cursor)
        except Exception as e:
            print(f"[WARNING] Error occured for '{module_code}' with Quiz IDs: {quiz_ids}")
            print(f"Reason: {e}")
            continue
        
        # Update module node in graph database with engagement analysis
        graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) 
                    where m.code = '{module_code}' and c.id = {course_id}
                    set m.statistics = coalesce(m.statistics,'') + '{module_stats}\n\n'""")

        # Process individual learner statistics for module participation
        for username in learner_stats:
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) 
                    MATCH (l:LEARNER)-[]->(c) 
                    where l.username = '{username}' and m.code = '{module_code}' and c.id = {course_id}
                    MERGE (l)-[r:PARTICIPATE]->(m)
                    set r.engagement = coalesce(r.engagement,'') + '{learner_stats[username]}\n\n'""")

        # Process individual quiz statistics (Learner -> Quiz relationships)
        for quiz_id in quiz_ids:

            try:
                # Generate statistics for individual quiz
                learner_stats, learner_data, activity_stats, activity_data = generate_quiz_statistics(
                    df=df_engagement, pilot=pilot, module_id=None, quiz_ids=[quiz_id], course_id=course_id, graph=graph, moodle_settings=moodle_settings, cursor=cursor
                )
            except Exception as e:
                print(f"[WARNING] Error occured for Quiz ID: {quiz_id}")
                print(f"Reason: {e}")
                continue

            # Update activity node with engagement analysis
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY) 
                        where m.code = '{module_code}' and a.id = 'QUIZ:{quiz_id}' and c.id = {course_id}
                        set a.statistics = '{activity_stats}'""")

            # Update learner-activity relationships with individual engagement data
            for username in learner_stats:
                graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY)
                            where c.id = {course_id} and m.code = '{module_code}' and a.id = 'QUIZ:{quiz_id}'
                            MATCH (l:LEARNER)-[r]->(a)
                            where l.username = '{username}'
                            set r.engagement = '{learner_stats[username]}',
                                r.number_of_actions = {learner_data[username]['total_actions']},
                                r.number_of_submissions = {learner_data[username]['completed_attempts']},
                                r.number_of_attempts = {learner_data[username]['total_attempts']},
                                r.time = {learner_data[username]['total_time_spent']},
                                r.number_of_clicks = {learner_data[username]['total_clicks']},
                                r.avg_time_per_attempt = {learner_data[username]['avg_time_per_attempt']},
                                r.number_of_completed_attempts = {learner_data[username]['completed_attempts']}""")

#### Activity: ASSIGN

##### Include ASSIGN's description

In [ ]:
query = f"""SELECT 
    a.id AS assign_id,
    a.course AS course_id,
    a.name AS assign_name,
    a.intro AS assign_intro
FROM 
    mdl_assign a
WHERE
    a.course in {course_ids}"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
print("[INFO] Number of records: ", df.shape[0])
# Processing
df["assign_intro"] = df["assign_intro"].apply(html2text)

for idx in tqdm(df.index):
    assign_id = df.loc[idx]["assign_id"]
    assign_name = text_preprocess(df.loc[idx]["assign_name"])
    assign_intro = text_preprocess(df.loc[idx]["assign_intro"])
    course_id = df.loc[idx]['course_id']

    query = f"""MATCH (a:ACTIVITY)
                WHERE a.id = "ASSIGN:{assign_id}" AND a.organization = "{get_organization(course_id)}"
                SET a.title = "{assign_name}", a.description = "{assign_intro}"
             """

    graph.query(query)

print("[INFO] ASSIGN properties were updated")

##### Retrieve Rubrics

In [ ]:
# Step 1. Get learner's performance based on Rubric
query = f"""SELECT
    a.id AS assign_id,
    gi.itemid as itemid,
    a.name AS assign_name,
    a.course as course_id,
    rc.id AS rubric_criterion_id,
    rls.definition AS level_definition,
    grf.levelid AS levelid,
    rls.score AS level_score,
    rc.description AS description
FROM
    mdl_assign AS a
JOIN
    mdl_course_modules AS cm ON cm.instance = a.id AND cm.module = (
        SELECT id FROM mdl_modules WHERE name = 'assign'
    )
JOIN
    mdl_context AS ctx ON ctx.instanceid = cm.id AND ctx.contextlevel = 70
JOIN
    mdl_grading_areas AS ga ON ga.contextid = ctx.id
JOIN
    mdl_grading_definitions AS gd ON gd.areaid = ga.id AND gd.method = 'rubric'
JOIN
    mdl_gradingform_rubric_criteria AS rc ON rc.definitionid = gd.id
JOIN
    mdl_gradingform_rubric_levels AS rls ON rls.criterionid = rc.id
JOIN
    mdl_grading_instances AS gi ON gi.definitionid = gd.id
JOIN
    mdl_gradingform_rubric_fillings AS grf ON grf.instanceid = gi.id AND grf.criterionid = rc.id
WHERE
    grf.levelid = rls.id and a.course in {course_ids}
ORDER BY 
     a.id;"""
# Retrieve user responses from the Moodle database
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
df_rubrics = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).drop_duplicates()
df_rubrics["level_definition"] = df_rubrics["level_definition"].apply(lambda x: x.replace('\r', '').replace('\n', ''))
print("[INFO] (Rubrics) Number of records: ", df_rubrics.shape[0])
# Normalization
df_rubrics = pd.merge(df_rubrics, df_rubric_scores,on=["rubric_criterion_id"], how="left")
df_rubrics['level_score'] /= df_rubrics['max_level_score']
# Include 4Cs
df_rubrics['metric'] = df_rubrics['description'].apply(assignment_4C)

# Step 2. Retrieve grades
query = "SELECT assignment AS assign_id, userid AS user_id, id AS itemid, grade AS grade from mdl_assign_grades"
# Retrieve user responses from the Moodle database
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
df_grades = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).drop_duplicates().dropna(ignore_index=True)
print("[INFO] (Learner grades) Number of records: ", df_grades.shape[0])

# Step 3. Include Rubrics along with the corresponding grade & Include 4C grades
for metric in available_metrics: 
    if metric == "Cognitive": continue
    df_grades[metric] = None
df_grades['Creativity_reasoning'] = None
df_grades['Collaboration_reasoning'] = None
df_grades['Critical thinking_reasoning'] = None
df_grades['Communication_reasoning'] = None
df_grades['Cognitive_reasoning'] = None
for idx in df_grades.index:
    # Get rubrics per metric
    temp_df = df_rubrics[df_rubrics['itemid'] == df_grades['itemid'][idx]][['level_definition', 'metric']].groupby('metric').agg(list).reset_index()
    temp_df['level_definition'] = temp_df['level_definition'].apply(lambda L: text_preprocess(" ".join([f"{x}" for x in L if len(x) > 10])))
    temp_df['level_definition'] = temp_df['level_definition'].apply(lambda x: x.replace("'",""))
    rubrics = dict(zip(temp_df['metric'].str.strip(), temp_df['level_definition'].str.strip()))
    
    # Get 4Cs grades
    d = df_rubrics[(df_rubrics['itemid'] == df_grades['itemid'][idx])][['metric', 'level_score']].groupby('metric').mean().to_dict()['level_score']
    
    for metric, grade in d.items():
        df_grades.loc[idx, f"{metric}_reasoning"] = rubrics[metric]
        if metric == "Cognitive": continue
        df_grades.loc[idx, metric] = 100.0 * grade

##### Assign Learners to ASSIGNs

In [ ]:
query = f"""SELECT 
    s.userid AS user_id,
    a.id AS assign_id,
    a.course AS course_id,
    MAX(g.grade) AS Cognitive,
    a.grade as assign_max_grade
FROM 
    mdl_assign_submission s
JOIN 
    mdl_assign a ON s.assignment = a.id
LEFT JOIN 
    mdl_assign_grades g ON a.id = g.assignment AND s.userid = g.userid
WHERE
    a.course IN {course_ids} 
GROUP BY 
    s.userid, a.id, g.grade;
"""
# Probably this constraint 'AND l.action IN ('submitted', 'viewed', 'updated', 'graded', 'removed, 'feedback')' is not need

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df = df[df["user_id"].isin(learner_ids)]
print("[INFO] Number of records: ", df.shape[0])
# Include rubrics
df = pd.merge(df, df_grades[['user_id', 'assign_id', 'Creativity', 'Critical thinking', 'Collaboration', 'Communication', 'Creativity_reasoning', 'Collaboration_reasoning', 'Critical thinking_reasoning', 'Communication_reasoning', 'Cognitive_reasoning']], on=['user_id', 'assign_id'], how='left')
# Preprocess
for metric in available_metrics:
    df[metric] = df[metric].astype('float')
    df[metric] = df[metric].apply(lambda x: x if isnan(x) else round(x,1))
df["assign_max_grade"] = df["assign_max_grade"].apply(lambda x: x if x > 0 else None).astype('float')
df["Cognitive"] = df["Cognitive"].apply(lambda x: x if x >= 0 else NaN)
df["Cognitive"] = df.apply(lambda row: row['Cognitive'] if isnan(row['assign_max_grade']) else round(100.0*row['Cognitive']/row['assign_max_grade'], 1), axis=1)

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    assign_id = df.loc[idx]["assign_id"]
    course_id = df.loc[idx]['course_id']
    # Include metrics & their corresponding rubrics
    metrics_input, rubrics = [], []
    for metric in available_metrics:
        if df.loc[idx][metric] and not isnan(df.loc[idx][metric]):
            metrics_input += [f"{metric.replace(' ','_')}_grade:{df.loc[idx][metric]}"]
            if df.loc[idx][f"{metric}_reasoning"]:
                rubrics += [f"{metric.replace(' ','_')}_reasoning:'{df.loc[idx][f'{metric}_reasoning']}'"]
    features = ", ".join(metrics_input + rubrics)
          
    query = f"""MATCH (l:LEARNER) 
                WHERE l.user_id={user_id} AND l.organization = "{get_organization(course_id)}"
                MATCH (a:ACTIVITY) 
                WHERE a.id="ASSIGN:{assign_id}" AND a.organization = "{get_organization(course_id)}"
            MERGE (l)-[:PARTICIPATE {{ {features} }}]->(a)"""
    graph.query(query)   
    
print("[INFO] Learners were assigned to ASSIGNs")

##### Engagement information

In [ ]:
query = f"""SELECT 
    s.userid AS user_id,
    u.username AS username,
    a.id AS assign_id,
    a.course AS course_id,
    l.action as action,
    l.timecreated AS action_time_unix,
    FROM_UNIXTIME(l.timecreated) AS action_time_readable
FROM 
    mdl_assign_submission s
JOIN 
    mdl_assign a ON s.assignment = a.id
LEFT JOIN 
    mdl_logstore_standard_log l ON l.objectid = s.id AND l.userid = s.userid AND l.component = 'mod_assign'
LEFT JOIN
    mdl_user u ON u.id = s.userid
WHERE
    a.course IN {course_ids}
ORDER BY l.timecreated
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(
    query=query, moodle_settings=moodle_settings, cursor=cursor
)
# Convert records to DataFrame
df_engagement = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description]).dropna()
# Pre-processing
df_engagement = df_engagement[df_engagement["user_id"].isin(learner_ids)]
print("[INFO] Number of records: ", df_engagement.shape[0])

In [ ]:
from utils.engagement_assign import generate_assignment_statistics
import warnings
warnings.filterwarnings('ignore')

# Iterate through each course in the courses_info data structure
for course_item in courses_info:
    # Extract course ID from current course item
    course_id = course_item["course_id"]
    print(f"[INFO] Course ID: {course_id}")
    if course_id in (2, 4, 12):
        pilot = "IASIS"
    elif course_id in (10, 11):
        pilot = "EASD"
    elif course_id in (16, 33):
        pilot = "UPAT"
    elif course_id == 24:
        pilot = "KTU"
    elif course_id == 34:
        pilot = "Demo"        
    else:
        raise ValueError(f"Course with ID: {course_id} does not belong to any pilot")

    # Process each module within the current course
    for module_item in tqdm(course_item["modules"]):
        # Extract module ID and initialize empty list for ASSIGN IDs
        module_code, assign_ids = module_item["code"], []

        # Filter activity IDs to find only Assign activities
        for id in module_item["activity_ids"]:
            if "ASSIGN" in id:
                # Extract numeric assign ID from the string format (e.g., "ASSIGN:123" -> 123)
                assign_ids.append(int(id.split(":")[-1]))

        # Process module-level assign statistics if assignments exist
        if not assign_ids: continue        
        try:
            # Generate statistics for all assignments in this module
            learner_stats, learner_data, module_stats, module_data = generate_assignment_statistics(df=df_engagement, pilot=pilot, module_id=module_code, assign_ids=assign_ids, course_id=course_id, graph=graph, moodle_settings=moodle_settings, cursor=cursor)
        except Exception as e:
            print(f"[WARNING] Error occured for '{module_code}' with Assign IDs: {assign_ids}")
            print(f"Reason: {e}")
            continue

        # Update module node in graph database with engagement analysis
        graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) 
                    where m.code = '{module_code}' and c.id = {course_id}
                    set m.statistics = coalesce(m.statistics,'') + '{module_stats}\n\n'""")

        # Process individual learner statistics for module participation
        for username in learner_stats:
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) 
                    MATCH (l:LEARNER)-[]->(c) 
                    where l.username = '{username}' and m.code = '{module_code}' and c.id = {course_id}
                    MERGE (l)-[r:PARTICIPATE]->(m)
                    set r.engagement = coalesce(r.engagement,'') + '{learner_stats[username]}\n\n'""")

        # Process individual assign statistics (Learner -> Assign relationships)
        for assign_id in assign_ids:

            try:
                # Generate statistics for individual Assign
                learner_stats, learner_data, activity_stats, activity_data = generate_assignment_statistics(
                    df=df_engagement, pilot=pilot, module_id=None, assign_ids=[assign_id], course_id=course_id, graph=graph, moodle_settings=moodle_settings, cursor=cursor
                )
            except Exception as e:
                print(f"[WARNING] Error occured for Assign ID: {assign_id}")
                print(f"Reason: {e}")
                continue

            # Update activity node with engagement analysis
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY) 
                        where m.code = '{module_code}' and a.id = 'ASSIGN:{assign_id}' and c.id = {course_id}
                        set a.statistics = '{activity_stats}'""")
            
            # Update learner-activity relationships with individual engagement data
            for username in learner_stats:
                graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY)
                            where c.id = {course_id} and m.code = '{module_code}' and a.id = 'ASSIGN:{assign_id}'
                            MATCH (l:LEARNER)-[r]->(a)
                            where l.username = '{username}'
                            set r.engagement = '{learner_stats[username]}',
                                r.time = {learner_data[username]['total_time_spent']},
                                r.submitted = {1 if learner_data[username]['submitted_sessions'] > 0 else 0},
                                r.number_of_clicks = {learner_data[username]['total_clicks']},
                                r.number_of_actions = {learner_data[username]['total_actions']},
                                r.avg_time_per_attempt = {learner_data[username]['avg_time_per_session']}""")


#### Include performance for each learner to the corresponding activity

In [ ]:
# Retrieve grades for all learners and activities
query = f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY)
            where c.id in {list(course_ids)}
            match (l:LEARNER)-[r:PARTICIPATE]->(a)
            return c.id as course_id, m.code as module_id, a.id as activity_id, l.username as username, 
            r.Cognitive_grade as Cognitive_grade, r.Creativity_grade as Creativity_grade, r.Collaboration_grade as Collaboration_grade, r.Communication_grade as Communication_grade, r.Critical_thinking_grade as Critical_thinking_grade,
            r.Cognitive_reasoning as Cognitive_reasoning, r.Creativity_reasoning as Creativity_reasoning, r.Collaboration_reasoning as Collaboration_reasoning, r.Communication_reasoning as Communication_reasoning, r.Critical_thinking_reasoning as Critical_thinking_reasoning
"""
response = graph.query(query)
print("[INFO] Number of retrieved pairs (learner, activity): ", len(response))

# Assign performance for each learner to the corresponding activity
for item in tqdm(response):
    course_id = item['course_id']
    module_id = item['module_id']
    activity_id = item['activity_id']
    username = item['username']
    performance_text = ""
    for metric in available_metrics:
        metric = metric.replace(" ", "_")
        if item[f'{metric}_grade']:
            performance_text += f"**{metric} grade:** {item[f'{metric}_grade']}/100\n"
            if item[f'{metric}_reasoning']:
                performance_text += f"**Elaboration:** {item[f'{metric}_reasoning']}\n"
            else:
                performance_text += "**Elaboration:** -\n"

    query = f"""MATCH (c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY)
            WHERE c.id = {course_id} AND m.code = '{module_id}' AND a.id = '{activity_id}' 
            MATCH (l:LEARNER)-[r:PARTICIPATE]->(a)
            SET r.performance = '{performance_text}'"""
    graph.query(query)


### Learner's evaluation

#### Include Evaluation-Questionnaires

In [ ]:
query = f"""SELECT cm.course AS course_id,
    cs.id AS section_id,
    cs.section AS section_name,
    cs.name AS title,
    fi.feedback AS questionnaire,
    fi.id AS feedback_item_id, 
    fi.name AS feedback_item_name, 
    fc.userid as user_id, 
    u.username AS username,    
    fv.value AS user_response
FROM mdl_feedback_item fi
JOIN mdl_feedback f ON fi.feedback = f.id
JOIN mdl_course_modules cm ON cm.instance = f.id
JOIN mdl_course_sections cs ON cs.id = cm.section
JOIN mdl_feedback_completed fc ON fc.feedback = f.id
JOIN mdl_feedback_value fv ON fv.completed = fc.id AND fv.item = fi.id
JOIN mdl_user u ON u.id = fc.userid
WHERE cm.module = (SELECT id FROM mdl_modules WHERE name = 'feedback') 
    AND cm.course in {course_ids}
    AND fv.value != ''
    AND fv.value IS NOT NULL
ORDER BY cm.course, cs.id, fi.id, fc.userid
"""
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
# Pre-processing
df = df[df["user_id"].isin(learner_ids)]
print("[INFO] Number of records: ", df.shape[0])

In [ ]:
for questionnaire_id in tqdm(df['questionnaire'].unique()):
    if df[df['questionnaire'] == questionnaire_id]['feedback_item_id'].unique().size == 1:
        
        for idx in df[df['questionnaire'] == questionnaire_id][['course_id', 'section_id', 'username', 'user_response']].index:
            course_id = df.loc[idx]['course_id']
            module_id = df.loc[idx]['section_id']
            username = df.loc[idx]['username']
            user_response = int(df.loc[idx]['user_response'])
            
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE)
                        WHERE c.id = {course_id} and m.id = {module_id}
                        MATCH (l:LEARNER) 
                        where l.username = '{username}' AND l.organization = "{get_organization(course_id)}"
                        MERGE (l)-[r:PARTICIPATE]->(m)
                        SET r.evaluation='{d_eval_questionnaire[user_response][get_language(course_id)]}'""")

#### Update MODULE statistics

In [ ]:
from collections import Counter
from utils.translations import d_eval_questionnaire

for item in courses_info:
    course_id = item['course_id']
    language = get_language(course_id)
    for iitem in item['modules']:
        module_code = iitem['code']
        evaluation_stats = graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) where c.id = {course_id} and m.code = '{module_code}'
                                           MATCH (l:LEARNER)-[r]->(m) return collect(r.evaluation) as evaluation_stats""")[0]['evaluation_stats']

        if evaluation_stats:
            print(f"Course ID: {course_id} - {module_code}")
            # Count frequencies
            total = len(evaluation_stats)
            counts = Counter(evaluation_stats)

            # Calculate and print percentages in the correct order            
            if language == "greek":
                module_stats  = "## Αντιλήψεις Μαθητών για τη Διανοητική Απαίτηση του Μαθήματος\n\n"
            elif language == "serbian":
                module_stats  = "## Percepcije učenika o kognitivnim zahtevima nastave\n\n"
            else:
                module_stats = "## Students Perceptions of the Cognitive Demands of the Course\n\n"

            for category in [value[language] for i,value in d_eval_questionnaire.items()]:
                percentage = (counts[category] / total) * 100 if category in counts else 0
                module_stats += f"- {category}: {percentage:.1f}%\n"
            
            graph.query(f"""MATCH (c:COURSE)-[]->(m:MODULE) where c.id = {course_id} and m.code = '{module_code}' 
                            set m.statistics = coalesce(m.statistics,'') + '{module_stats}\n\n',
                            m.evaluation = '{module_stats}'""")

### Sign learners to activities if not graded

In [ ]:
for course_id in course_ids:
    query = f"""MATCH (l:LEARNER)-[]->(c:COURSE)-[]->(m:MODULE)-[]->(a:ACTIVITY)
                where c.id = {course_id}
                MATCH (l)-[r:PARTICIPATE]->(a)
                SET r.status = 'complete'"""
    graph.query(query)

In [ ]:
# Ensure all learners have PARTICIPATE relationships with all activities in their courses
for course_id in course_ids:
    # Get learners' usernames
    usernames = graph.query(f"""MATCH (l:LEARNER)-[]->(c:COURSE) where c.id = {course_id} return collect(l.username) as usernames""")[0]['usernames']

    # Get activity IDs with learners in a single query
    activities_IDs = [item['activity_id'] for item in graph.query(f"""
        MATCH (l:LEARNER)-[]->(c:COURSE)-[]-(m:MODULE)-[]->(a:ACTIVITY)
        WHERE c.id = {course_id}
        MATCH (l)-[:PARTICIPATE]->(a)
        WITH a.id as activity_id, count(distinct l) as number_of_learners
        WHERE number_of_learners > 0
        RETURN activity_id
    """)]
    
    # Create all relationships in ONE query using UNWIND
    if usernames and activities_IDs:
        graph.query(f"""
            UNWIND {usernames} AS username
            UNWIND {activities_IDs} AS activity_id
            MATCH (l:LEARNER {{username: username}})-[]->(c:COURSE)
            MATCH (c:COURSE)-[]-(m:MODULE)-[]->(a:ACTIVITY {{id: activity_id}})
            WHERE c.id = {course_id}
            AND NOT EXISTS((l)-[:PARTICIPATE]->(a))
            MERGE (l)-[:PARTICIPATE {{status: 'incomplete'}}]->(a)
        """)

### Resources

#### Book

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    u.id AS book_id,
    u.name AS book_name,
    u.intro AS description,
    CONCAT('{moodle_url}/mod/book/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_book u ON cm.instance = u.id
    WHERE m.name = 'book' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
print("[INFO] Number of records: ", df.shape[0])
# Preprocess
df['description'] = df['description'].apply(html2text)

for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    book_id = f"BOOK:{df.loc[idx]['book_id']}"
    book_title = df.loc[idx]['book_name']
    book_description = df.loc[idx]['description']
    URL = df.loc[idx]['URL']
    
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (r:RESOURCE {{id:"{book_id}", title: "{book_title}", description:"{book_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    MERGE (m)-[:HAS_RESOURCE]->(r)
    """
    graph.query(query)

print("[INFO] Resource:BOOK were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    book.id as book_id,
    book.course AS course_id
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_book book ON l.objectid = book.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'book'
    AND book.course IN {course_ids};"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    book_id = f"BOOK:{df.loc[idx]['book_id']}"
    course_id = df.loc[idx]['course_id']
    
    query = f"""MATCH (l:LEARNER) 
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{book_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)

print("[INFO] Learners which studied the Resource:BOOK were mapped")

#### Workshop

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    u.id AS workshop_id,
    u.name AS workshop_name,
    u.intro AS description,
    CONCAT('{moodle_url}/mod/workshop/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_workshop u ON cm.instance = u.id
    WHERE m.name = 'workshop' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
print("[INFO] Number of records: ", df.shape[0])
# Preprocess
df['description'] = df['description'].apply(html2text)

for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    workshop_id = f"WORKSHOP:{df.loc[idx]['workshop_id']}"
    workshop_title = df.loc[idx]['workshop_name']
    workshop_description = df.loc[idx]['description']
    URL = df.loc[idx]['URL']
    
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (r:RESOURCE {{id:"{workshop_id}", title: "{workshop_title}", description:"{workshop_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    MERGE (m)-[:HAS_RESOURCE]->(r)
    """
    graph.query(query)

print("[INFO] Resource:WORKSHOP were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    workshop.id as workshop_id,
    workshop.course AS course_id
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_workshop workshop ON l.objectid = workshop.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'workshop'
    AND workshop.course IN {course_ids};"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    workshop_id = f"WORKSHOP:{df.loc[idx]['workshop_id']}"
    course_id = df.loc[idx]['course_id']
    
    query = f"""MATCH (l:LEARNER) 
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{workshop_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)

print("[INFO] Learners which studied the Resource:WORKSHOP were mapped")

#### URL

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    u.id AS url_id,
    u.name AS url_name,
    u.externalurl AS external_url,
    CONCAT('{moodle_url}/mod/url/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_url u ON cm.instance = u.id
    WHERE m.name = 'url' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
print("[INFO] Number of records: ", df.shape[0])


for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    url_title = df.loc[idx]['url_name']
    url_description = df.loc[idx]['external_url']
    url_id = f"URL:{df.loc[idx]['url_id']}"
    URL = df.loc[idx]['URL']
    
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (r:RESOURCE {{id:"{url_id}", title: "{url_title}", description:"{url_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    MERGE (m)-[:HAS_RESOURCE]->(r)
    """
    graph.query(query)

print("[INFO] Resource:URL were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    url.id as url_id,
    url.course AS course_id
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_url url ON l.objectid = url.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'url'
    AND url.course IN {course_ids};"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    url_id = f"URL:{df.loc[idx]['url_id']}"
    course_id = df.loc[idx]['course_id']
    
    query = f"""MATCH (l:LEARNER) 
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{url_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)

print("[INFO] Learners which studied the Resource:URL were mapped")

#### Page

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    p.id AS page_id,
    p.name AS page_name,
    p.content as page_intro,
    CONCAT('{moodle_url}/mod/page/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_page p ON cm.instance = p.id
    WHERE m.name = 'page' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
# Preprocess
df['page_intro'] = df['page_intro'].apply(html2text)
df['page_intro'] = df['page_intro'].apply(lambda x: x if len(x) > 0 else "-")
print("[INFO] Number of records: ", df.shape[0])


for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    page_title = df.loc[idx]['page_name']
    page_description = df.loc[idx]['page_intro']    
    page_id = f"PAGE:{df.loc[idx]['page_id']}"
    URL = df.loc[idx]['URL']  
        
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (r:RESOURCE {{id:"{page_id}", title:"{page_title}", description:"{page_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    MERGE (m)-[:HAS_RESOURCE]->(r)
    """
    graph.query(query)

print("[INFO] Resource:PAGE were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    p.id AS page_id,
    p.course AS course_id
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_page p ON l.objectid = p.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'page'
    AND p.course in {course_ids}
    """

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    page_id = f"PAGE:{df.loc[idx]['page_id']}"
    course_id = df.loc[idx]['course_id']
    query = f"""MATCH (l:LEARNER)
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{page_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)

print("[INFO] Learners which studied the Resources:PAGE were mapped")

#### Folder

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    f.id AS folder_id,
    f.name AS folder_name,
    f.intro as folder_intro,
    CONCAT('{moodle_url}/mod/folder/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_folder f ON cm.instance = f.id
    WHERE m.name = 'folder' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
# Preprocess
df['folder_intro'] = df['folder_intro'].apply(html2text)
df['folder_intro'] = df['folder_intro'].apply(lambda x: x if len(x) > 0 else "-")
print("[INFO] Number of records: ", df.shape[0])


for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    folder_title = df.loc[idx]['folder_name']
    folder_description = df.loc[idx]['folder_intro']   
    folder_id = f"FOLDER:{df.loc[idx]['folder_id']}"
    URL = df.loc[idx]['URL']  
        
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (r:RESOURCE {{id:"{folder_id}", title:"{folder_title}", description:"{folder_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    MERGE (m)-[:HAS_RESOURCE]->(r)
    """
    graph.query(query)

print("[INFO] Resource:FOLDER were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    u.username AS username,
    f.id AS folder_id,
    f.course AS course_id    
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_folder f ON l.objectid = f.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'folder'
    AND f.course in {course_ids};
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    folder_id = f"FOLDER:{df.loc[idx]['folder_id']}"
    course_id = df.loc[idx]['course_id']
    
    query = f"""MATCH (l:LEARNER)
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{folder_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)

print("[INFO] Learners which studied the Resources:FOLDER were mapped")

#### Glossary

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    g.id AS glossary_id,
    g.name AS glossary_name,
    g.intro as glossary_intro,
    CONCAT('{moodle_url}/mod/glossary/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_glossary g ON cm.instance = g.id
    WHERE m.name = 'glossary' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
# Preprocess
df['glossary_intro'] = df['glossary_intro'].apply(html2text)
df['glossary_intro'] = df['glossary_intro'].apply(lambda x: x if len(x) > 0 else "-")
print("[INFO] Number of records: ", df.shape[0])


for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    glossary_title = df.loc[idx]['glossary_name']
    glossary_description = df.loc[idx]['glossary_intro']   
    glossary_id = f"GLOSSARY:{df.loc[idx]['glossary_id']}"
    URL = df.loc[idx]['URL']  
        
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (r:RESOURCE {{id:"{glossary_id}", title:"{glossary_title}", description:"{glossary_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    MERGE (m)-[:HAS_RESOURCE]->(r)
    """
    graph.query(query)

print("[INFO] Resource:GLOSSARY were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    u.username AS username,
    g.course AS course_id,    
    g.id AS glossary_id
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_glossary g ON l.objectid = g.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'glossary'
    AND g.course in {course_ids};
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    glossary_id = f"GLOSSARY:{df.loc[idx]['glossary_id']}"
    course_id = df.loc[idx]['course_id']
    
    query = f"""MATCH (l:LEARNER)
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{glossary_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)
print("[INFO] Learners which studied the Resources:GLOSSARY were mapped")

#### Feedback

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    f.id AS feedback_id,
    f.name AS feedback_name,
    f.intro as feedback_intro,
    CONCAT('{moodle_url}/mod/feedback/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_feedback f ON cm.instance = f.id
    WHERE m.name = 'feedback' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
# Preprocess
df['feedback_intro'] = df['feedback_intro'].apply(html2text)
df['feedback_intro'] = df['feedback_intro'].apply(lambda x: x if len(x) > 0 else "-")
print("[INFO] Number of records: ", df.shape[0])


for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    feedback_title = df.loc[idx]['feedback_name']
    feedback_description = df.loc[idx]['feedback_intro'] 
    feedback_id = f"FEEDBACK:{df.loc[idx]['feedback_id']}"
    URL = df.loc[idx]['URL']  
        
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (r:RESOURCE {{id:"{feedback_id}", title:"{feedback_title}", description:"{feedback_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    MERGE (m)-[:HAS_RESOURCE]->(r)
    """
    graph.query(query)

print("[INFO] Resource:FEEDBACK were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    u.username AS username,
    f.course AS course_id,    
    f.id AS feedback_id
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_feedback f ON l.objectid = f.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'feedback'
    AND f.course in {course_ids};
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    feedback_id = f"FEEDBACK:{df.loc[idx]['feedback_id']}"

    query = f"""MATCH (l:LEARNER)
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{feedback_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)
print("[INFO] Learners which studied the Resources:FEEDBACK were mapped")

#### H5P

In [ ]:
query = f"""SELECT 
    cs.course AS course_id,
    cs.id AS module_id,
    h.id AS h5p_id,
    h.name AS h5p_name,
    h.intro as h5p_intro,
    CONCAT('{moodle_url}/mod/h5p/view.php?id=', cm.id) AS URL
    
    FROM mdl_course_sections cs
    JOIN mdl_course_modules cm ON cs.id = cm.section
    JOIN mdl_modules m ON cm.module = m.id
    JOIN mdl_h5pactivity h ON cm.instance = h.id
    WHERE m.name = 'h5pactivity' and cs.course in {course_ids};
    
    ORDER BY cs.course, cs.section;
"""
# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
# Preprocess
df['h5p_intro'] = df['h5p_intro'].apply(html2text)
df['h5p_intro'] = df['h5p_intro'].apply(lambda x: x if len(x) > 0 else "-")
print("[INFO] Number of records: ", df.shape[0])


for idx in tqdm(df.index):
    course_id = df.loc[idx]['course_id']
    module_id = df.loc[idx]['module_id']
    h5p_title = df.loc[idx]['h5p_name']
    h5p_description = df.loc[idx]['h5p_intro']   
    h5p_id = f"H5P:{df.loc[idx]['h5p_id']}"
    URL = df.loc[idx]['URL']  
        
    query = f"""MATCH (c:COURSE)-[:HAS_MODULE]->(m:MODULE)
    WHERE c.id = {course_id} and m.id = {module_id}
    MERGE (m)-[:HAS_RESOURCE]->(r:RESOURCE {{id:"{h5p_id}", title:"{h5p_title}", description:"{h5p_description}", organization:"{get_organization(course_id)}", url:"{URL}"}})
    """
    graph.query(query)

print("[INFO] Resource:H5P were imported")

In [ ]:
query = f"""SELECT DISTINCT
    u.id AS user_id,
    u.username AS username,
    h5p.course as course_id,
    h5p.id AS h5p_id
FROM
    mdl_logstore_standard_log l
JOIN
    mdl_user u ON l.userid = u.id
JOIN
    mdl_h5pactivity h5p ON l.objectid = h5p.id
WHERE
    l.action = 'viewed'
    AND l.objecttable = 'h5pactivity'
    AND h5p.course IN {course_ids};
"""

# Fetch records
rows, cursor = retrieve_data_from_MOODLE(query=query, moodle_settings=moodle_settings, cursor=cursor)
# Convert records to DataFrame
df = pd.DataFrame(rows, columns=[desc[0] for desc in cursor.description])
df.drop_duplicates(inplace=True)
print("[INFO] Number of records: ", df.shape[0])

for idx in tqdm(df.index):
    user_id = df.loc[idx]["user_id"]
    h5p_id = f"H5P:{df.loc[idx]['h5p_id']}"
    course_id = df.loc[idx]['course_id']
    
    query = f"""MATCH (l:LEARNER)
    WHERE l.user_id = {user_id} AND l.organization = "{get_organization(course_id)}"
    MATCH (r:RESOURCE)
    WHERE r.id = "{h5p_id}" AND r.organization = "{get_organization(course_id)}"
    MERGE (l)-[:STUDY]->(r)
    """
    graph.query(query)

print("[INFO] Learners which studied the Resources:H5P were mapped")

In [ ]:
if connection.is_connected():
    cursor.close()
    connection.close()
    print("[INFO] MySQL connection is closed")